<a href="https://colab.research.google.com/github/AmitabhDey-byte/ANN/blob/main/BIJA_130M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os,re,gc,json,math,time,random,hashlib,shutil,csv,gzip,unicodedata
from pathlib import Path
from dataclasses import dataclass,asdict
from collections import defaultdict
from typing import Dict,List,Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.checkpoint import checkpoint
from datasets import load_dataset
from tokenizers import Tokenizer,models,trainers,pre_tokenizers,decoders,normalizers
from transformers import PreTrainedTokenizerFast
from safetensors.torch import save_model
from tqdm.auto import tqdm
import boto3
from botocore import UNSIGNED
from botocore.config import Config


In [2]:
!pip install boto3
!pip install botocore

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.9 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
TEST_RUN=True
ROOT="/content/drive/MyDrive/BIJA-130M"
MODE="test" if TEST_RUN else "full"
BASE_DIR=f"{ROOT}/{MODE}" if TEST_RUN else ROOT
DATA_DIR=f"{BASE_DIR}/data"
TOKENIZER_DIR=f"{BASE_DIR}/tokenizer"
CHECKPOINT_DIR=f"{BASE_DIR}/checkpoints"
METRIC_DIR=f"{BASE_DIR}/metrics"
FINAL_DIR=f"{BASE_DIR}/final"

for p in [DATA_DIR,TOKENIZER_DIR,CHECKPOINT_DIR,METRIC_DIR,FINAL_DIR]:
    os.makedirs(p,exist_ok=True)

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")

In [6]:
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
VOCAB_SIZE=32768
CONTEXT_LENGTH=1024
TOTAL_TRAIN_TOKENS=4_400_000_000

SOURCE_TOKENS={
    "fineweb":2_200_000_000,
    "dclm":880_000_000,
    "cosmopedia":660_000_000,
    "finemath":264_000_000,
    "wikipedia":176_000_000,
    "stack_python":110_000_000,
    "stack_javascript":22_000_000,
    "stack_typescript":22_000_000,
    "stack_java":33_000_000,
    "stack_c":11_000_000,
    "stack_cpp":11_000_000,
    "stack_rust":11_000_000
}

SOURCE_SPECS={"fineweb":{"name":"HuggingFaceTB/smollm-corpus","config":"fineweb-edu-dedup","kind":"text"},
    "dclm":{"name":"mlfoundations/dclm-baseline-1.0-parquet","config":None,"kind":"text"},
    "cosmopedia":{"name":"HuggingFaceTB/smollm-corpus","config":"cosmopedia-v2","kind":"text"},
    "finemath":{"name":"HuggingFaceTB/finemath","config":"finemath-4plus","kind":"text"},
    "wikipedia":{"name":"wikimedia/wikipedia","config":"20231101.en","kind":"text"},
    "stack_python":{"name":"HuggingFaceTB/stack-edu","config":"Python","kind":"stack"},
    "stack_javascript":{"name":"HuggingFaceTB/stack-edu","config":"JavaScript","kind":"stack"},
    "stack_typescript":{"name":"HuggingFaceTB/stack-edu","config":"TypeScript","kind":"stack"},
    "stack_java":{"name":"HuggingFaceTB/stack-edu","config":"Java","kind":"stack"},
    "stack_c":{"name":"HuggingFaceTB/stack-edu","config":"C","kind":"stack"},
    "stack_cpp":{"name":"HuggingFaceTB/stack-edu","config":"Cpp","kind":"stack"},
    "stack_rust":{"name":"HuggingFaceTB/stack-edu","config":"Rust","kind":"stack"}}

STAGE1_SOURCE_TOKENS={"fineweb":1_892_000_000,
    "dclm":792_000_000,
    "cosmopedia":440_000_000,
    "finemath":158_400_000,
    "wikipedia":105_600_000,
    "stack_python":66_000_000,
    "stack_javascript":13_200_000,
    "stack_typescript":13_200_000,
    "stack_java":19_800_000,
    "stack_c":6_600_000,
    "stack_cpp":6_600_000,
    "stack_rust":6_600_000
}

STAGE2_SOURCE_TOKENS={k:SOURCE_TOKENS[k]-STAGE1_SOURCE_TOKENS[k]
    for k in SOURCE_TOKENS
}

if TEST_RUN:
    TRAIN_TARGETS={k:max(100_000,int(v/TOTAL_TRAIN_TOKENS*5_000_000))
        for k,v in SOURCE_TOKENS.items()}
    VAL_TARGETS={k:20_000 for k in SOURCE_TOKENS}
    SHARD_TOKENS=500_000
    TOKENIZER_SAMPLE_CHARS=20_000_000
    RUN_TOTAL_TOKENS=5_000_000
else:
    TRAIN_TARGETS=SOURCE_TOKENS.copy()
    VAL_TARGETS={k:max(100_000,min(1_000_000,v//1000))
        for k,v in SOURCE_TOKENS.items()}
    SHARD_TOKENS=200_000_000
    TOKENIZER_SAMPLE_CHARS=1_000_000_000
    RUN_TOTAL_TOKENS=TOTAL_TRAIN_TOKENS

STAGE_SWITCH_TOKENS=int(RUN_TOTAL_TOKENS*0.8)
SPECIAL_TOKENS=["<|pad|>","<|bos|>","<|eos|>","<|unk|>","<|doc|>"]

print(DEVICE,MODE,sum(TRAIN_TARGETS.values()),RUN_TOTAL_TOKENS)

cuda test 5475000 5000000


In [7]:
S3=boto3.client("s3",
    config=Config(signature_version=UNSIGNED,connect_timeout=20,read_timeout=90,retries={"max_attempts":5}))

def clean_text(text:str,code:bool=False)->str:
    if not isinstance(text,str):
        return ""
    text=unicodedata.normalize("NFC",text.replace("\x00","").replace("\r\n","\n").replace("\r","\n"))
    if not code:
        text=re.sub(r"\n{4,}","\n\n\n",text)
    return text.strip()

def fetch_code(blob_id:str)->str:
    try:
        obj=S3.get_object(Bucket="softwareheritage",Key=f"content/{blob_id}")
        with gzip.GzipFile(fileobj=obj["Body"]) as f:
            return f.read().decode("utf-8",errors="ignore")
    except Exception:
        return ""

def load_stream(source:str,skip_rows:int=0,seed_offset:int=0):
    spec=SOURCE_SPECS[source]
    kwargs={
        "path":spec["name"],
        "split":"train",
        "streaming":True
    }
    if spec["config"] is not None:
        kwargs["name"]=spec["config"]
    ds=load_dataset(**kwargs)
    ds=ds.shuffle(
        seed=SEED+seed_offset,
        buffer_size=2000 if spec["kind"]=="stack" else 10000)
    if skip_rows:
        ds=ds.skip(skip_rows)
    return ds

def iter_source_rows(source:str,skip_rows:int=0,seed_offset:int=0):
    spec=SOURCE_SPECS[source]
    for row in load_stream(source,skip_rows,seed_offset):
        if spec["kind"]=="stack":
            if row.get("license_type")!="permissive":
                yield ""
                continue
            if int(row.get("length_bytes",0))<64:
                yield ""
                continue
            if int(row.get("length_bytes",0))>1_000_000:
                yield ""
                continue
            text=fetch_code(row.get("blob_id",""))
            yield clean_text(text,True)
        else:
            yield clean_text(row.get("text",""),False)

def tokenizer_text_iterator():
    for i,source in enumerate(SOURCE_SPECS):
        target=max(
            100_000,
            int(
                TOKENIZER_SAMPLE_CHARS*
                SOURCE_TOKENS[source]/
                TOTAL_TRAIN_TOKENS
            )
        )
        used=0
        for text in iter_source_rows(source,0,100+i):
            if len(text)<100:
                continue
            if len(text)>1_000_000:
                text=text[:1_000_000]
            yield text
            used+=len(text)
            if used>=target:
                break

In [8]:
TOKENIZER_JSON=f"{TOKENIZER_DIR}/tokenizer.json"

if not os.path.exists(TOKENIZER_JSON):
    tok=Tokenizer(models.BPE(unk_token="<|unk|>"))
    tok.normalizer=normalizers.NFC()
    tok.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder=decoders.ByteLevel()

    trainer=trainers.BpeTrainer(vocab_size=VOCAB_SIZE,min_frequency=2,special_tokens=SPECIAL_TOKENS,show_progress=True)

    tok.train_from_iterator(tokenizer_text_iterator(),trainer=trainer)

    tok.save(TOKENIZER_JSON)

    fast=PreTrainedTokenizerFast(tokenizer_object=tok,
        pad_token="<|pad|>",
        bos_token="<|bos|>",
        eos_token="<|eos|>",
        unk_token="<|unk|>",
        additional_special_tokens=["<|doc|>"],
        model_max_length=CONTEXT_LENGTH,
        clean_up_tokenization_spaces=False)

    fast.save_pretrained(TOKENIZER_DIR)

FAST_TOKENIZER=PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)

print(len(FAST_TOKENIZER),FAST_TOKENIZER.special_tokens_map)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.05k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/6.55k [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/14.8k [00:00<?, ?B/s]

32768 {'bos_token': '<|bos|>', 'eos_token': '<|eos|>', 'unk_token': '<|unk|>', 'pad_token': '<|pad|>'}


In [9]:
TOKENIZER=Tokenizer.from_file(TOKENIZER_JSON)

PAD_ID=TOKENIZER.token_to_id("<|pad|>")
BOS_ID=TOKENIZER.token_to_id("<|bos|>")
EOS_ID=TOKENIZER.token_to_id("<|eos|>")
UNK_ID=TOKENIZER.token_to_id("<|unk|>")
DOC_ID=TOKENIZER.token_to_id("<|doc|>")

assert [PAD_ID,BOS_ID,EOS_ID,UNK_ID,DOC_ID]==[0,1,2,3,4]
assert TOKENIZER.get_vocab_size()<=65535

def file_hash(path:str)->str:
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

TOKENIZER_HASH=file_hash(TOKENIZER_JSON)

print(TOKENIZER.get_vocab_size(),
    PAD_ID,
    BOS_ID,
    EOS_ID,
    UNK_ID,
    DOC_ID,TOKENIZER_HASH[:16])

32768 0 1 2 3 4 3a95578db5b0551c


In [10]:
class TokenShardWriter:
    def __init__(self,source:str):
        self.source=source
        self.dir=f"{DATA_DIR}/{source}"
        os.makedirs(self.dir,exist_ok=True)

        self.train_target=TRAIN_TARGETS[source]
        self.val_target=VAL_TARGETS[source]
        self.state_path=f"{self.dir}/state.json"

        self.state={"raw_rows_seen":0,
            "train_tokens":0,
            "val_tokens":0,
            "shard_index":0,
            "shard_pos":0,
            "completed":False
        }

        if os.path.exists(self.state_path):
            with open(self.state_path) as f:
                self.state.update(json.load(f))

        self.val_mm=None
        self.train_mm=None

        if not self.state["completed"]:
            self._open()

    def _open(self):
        if self.state["val_tokens"]<self.val_target:
            path=f"{self.dir}/val.bin"
            if not os.path.exists(path):
                mm=np.memmap(path,dtype=np.uint16,mode="w+",shape=(self.val_target,))
                mm.flush()
                del mm
            self.val_mm=np.memmap(path,dtype=np.uint16,mode="r+",shape=(self.val_target,))

        if self.state["train_tokens"]<self.train_target:
            base=self.state["train_tokens"]-self.state["shard_pos"]
            capacity=min(SHARD_TOKENS,self.train_target-base)
            path=f"{self.dir}/train_{self.state['shard_index']:05d}.bin"

            if not os.path.exists(path):
                mm=np.memmap(path,dtype=np.uint16,mode="w+",shape=(capacity,))
                mm.flush()
                del mm

            self.train_mm=np.memmap(path,dtype=np.uint16,mode="r+",shape=(capacity,))

    def _close_train(self):
        if self.train_mm is not None:
            self.train_mm.flush()
            del self.train_mm
            self.train_mm=None

    def _next_shard(self):
        self._close_train()
        self.state["shard_index"]+=1
        self.state["shard_pos"]=0

        if self.state["train_tokens"]<self.train_target:
            self._open()

    def write(self,ids):
        arr=np.asarray(ids,dtype=np.uint16)
        pos=0

        if self.state["val_tokens"]<self.val_target:
            n=min(len(arr),
                self.val_target-self.state["val_tokens"])
            a=self.state["val_tokens"]
            self.val_mm[a:a+n]=arr[:n]
            self.state["val_tokens"]+=n
            pos+=n

        while pos<len(arr) and self.state["train_tokens"]<self.train_target:
            room=len(self.train_mm)-self.state["shard_pos"]

            n=min(len(arr)-pos, room,self.train_target-self.state["train_tokens"])

            a=self.state["shard_pos"]
            self.train_mm[a:a+n]=arr[pos:pos+n]

            self.state["shard_pos"]+=n
            self.state["train_tokens"]+=n
            pos+=n

            if self.state["shard_pos"]==len(self.train_mm):
                if self.state["train_tokens"]<self.train_target:
                    self._next_shard()

    def save(self):
        if self.val_mm is not None:
            self.val_mm.flush()

        if self.train_mm is not None:
            self.train_mm.flush()

        tmp=self.state_path+".tmp"

        with open(tmp,"w") as f:
            json.dump(self.state,f)

        os.replace(tmp,self.state_path)

    def finish(self):
        self.state["completed"]=( self.state["train_tokens"]>=self.train_target and
            self.state["val_tokens"]>=self.val_target)

        self.save()

        if self.val_mm is not None:
            del self.val_mm
            self.val_mm=None

        self._close_train()

In [11]:
def tokenize_source(source:str):
    writer=TokenShardWriter(source)

    if writer.state["completed"]:
        print(source,"already complete")
        return

    raw=writer.state["raw_rows_seen"]
    pbar=tqdm(desc=source,unit="rows")

    try:
        for text in iter_source_rows(source,raw,500):
            raw+=1
            pbar.update(1)

            if len(text)>=100:
                try:
                    ids=[DOC_ID,*TOKENIZER.encode(text).ids,EOS_ID]
                    writer.write(ids)
                except Exception:
                    pass

            writer.state["raw_rows_seen"]=raw

            if raw%100==0:
                writer.save()
                pbar.set_postfix(train=writer.state["train_tokens"],val=writer.state["val_tokens"])

            if writer.state["train_tokens"]>=writer.train_target:
                if writer.state["val_tokens"]>=writer.val_target:
                    break

    except KeyboardInterrupt:
        writer.state["raw_rows_seen"]=raw
        writer.save()
        writer.finish()
        pbar.close()
        raise

    writer.state["raw_rows_seen"]=raw

    if writer.state["train_tokens"]<writer.train_target:
        writer.save()
        writer.finish()
        pbar.close()
        raise RuntimeError(f"{source} ended before its train token target")

    if writer.state["val_tokens"]<writer.val_target:
        writer.save()
        writer.finish()
        pbar.close()
        raise RuntimeError(f"{source} ended before its validation token target")

    writer.finish()
    pbar.close()

    print(source,writer.state)

In [12]:
SOURCES_TO_BUILD=list(SOURCE_SPECS)

for source in SOURCES_TO_BUILD:
    tokenize_source(source)

fineweb: 0rows [00:00, ?rows/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

fineweb {'raw_rows_seen': 2313, 'train_tokens': 2500000, 'val_tokens': 20000, 'shard_index': 4, 'shard_pos': 500000, 'completed': True}


dclm: 0rows [00:00, ?rows/s]

...

stack_c {'raw_rows_seen': 540, 'train_tokens': 100000, 'val_tokens': 20000, 'shard_index': 0, 'shard_pos': 100000, 'completed': True}


stack_cpp: 0rows [00:00, ?rows/s]

stack_cpp {'raw_rows_seen': 469, 'train_tokens': 100000, 'val_tokens': 20000, 'shard_index': 0, 'shard_pos': 100000, 'completed': True}


stack_rust: 0rows [00:00, ?rows/s]

stack_rust {'raw_rows_seen': 140, 'train_tokens': 100000, 'val_tokens': 20000, 'shard_index': 0, 'shard_pos': 100000, 'completed': True}


In [13]:
manifest={"mode":MODE,
    "vocab_size":TOKENIZER.get_vocab_size(),
    "context_length":CONTEXT_LENGTH,
    "tokenizer_hash":TOKENIZER_HASH,
    "sources":{}
}

for source in SOURCE_SPECS:
    files=sorted(str(p)
        for p in Path(f"{DATA_DIR}/{source}").glob("train_*.bin"))

    sizes=[os.path.getsize(path)//2
        for path in files]

    val_path=f"{DATA_DIR}/{source}/val.bin"
    val_size=os.path.getsize(val_path)//2

    manifest["sources"][source]={"files":files,
        "tokens":sum(sizes),
        "val_file":val_path,
        "val_tokens":val_size
    }

    assert sum(sizes)>=TRAIN_TARGETS[source]
    assert val_size>=VAL_TARGETS[source]

MANIFEST_PATH=f"{DATA_DIR}/manifest.json"

with open(MANIFEST_PATH,"w") as f:
    json.dump(manifest,f,indent=2)

DATA_MANIFEST_HASH=file_hash(MANIFEST_PATH)

print(DATA_MANIFEST_HASH[:16],
    {
        k:v["tokens"]
        for k,v in manifest["sources"].items()
    }
)

72ae4e59ee069944 {'fineweb': 2500000, 'dclm': 1000000, 'cosmopedia': 750000, 'finemath': 300000, 'wikipedia': 200000, 'stack_python': 125000, 'stack_javascript': 100000, 'stack_typescript': 100000, 'stack_java': 100000, 'stack_c': 100000, 'stack_cpp': 100000, 'stack_rust': 100000}
